PUMA Exploration 7 - Main

This file runs the experiments for Exploration 7.

Main points:
- Keep the old Exploration 6 natural data only for checking old results.
- Exp1 and Exp2 use new natural calibration and test data.
- Keep the base setup fixed: UNI2-h + Gaussian sigma=1.5 + Tier-A + A5 rank 8 + FOV96.
- Keep UNI2-h frozen.
- Do not do a large hyperparameter search.
- Check recall, precision, and false positives for each experiment.


0. Set the project paths


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys

PROJECT_ROOT = Path('/content/drive/MyDrive/Research/PUMA')

P7 = PROJECT_ROOT / 'Exploration_7_Causal_Bottleneck_and_Natural_Generalization'
P6 = PROJECT_ROOT / 'Exploration_6_Generalization_Scaling_and_Final_Gaussian_Model'
assert (P7 / 'README.md').is_file(), f'Exploration-7 folder not found: {P7}'
print('Exploration7:', P7)
print('Exploration6:', P6)

1. Install the required packages


In [ ]:
req = [
    x.strip() for x in (P7/'CODE/COMMON/requirements.txt').read_text().splitlines()
    if x.strip() and not x.strip().startswith('torch')
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *req], check=True)
os.environ['PYTHONPATH'] = str(P7/'CODE/COMMON/src') + os.pathsep + os.environ.get('PYTHONPATH','')
print('Dependencies ready. Torch is left as the Colab-provided CUDA build.')


2. Check the files and setup


In [ ]:
required = [
    P7/'README.md',
    P7/'CODE/COMMON/src/puma_exploration7/__init__.py',
    P7/'CODE/CONFIGS/00_PROTOCOL.json',
    P7/'INPUTS/TEMPLATES/PATHS_TEMPLATE.json',
]
missing = [str(x) for x in required if not x.exists()]
if missing:
    raise FileNotFoundError('Exploration-7 runtime files missing: ' + ', '.join(missing))
sys.path.insert(0, str(P7/'CODE/COMMON/src'))
import puma_exploration7
import torch, timm
if timm.__version__ != '1.0.20':
    raise RuntimeError(f'Expected timm==1.0.20, got {timm.__version__}')
print('Runtime versions:', {'python': sys.version.split()[0], 'torch': torch.__version__, 'timm': timm.__version__, 'cuda': torch.cuda.is_available()})
for cfg in sorted((P7/'CODE/CONFIGS').rglob('*.json')):
    json.loads(cfg.read_text())
print('Exploration-7 runtime preflight passed.')


3. Check the old Exploration 6 result

This part is only used to compare with the previous result.
Do not use it to tune Exploration 7.


In [ ]:
RUN_HISTORICAL_DIAGNOSTIC = True
if RUN_HISTORICAL_DIAGNOSTIC:
    result = P6/'COLAB_RESULTS/STAGE_I/LOCKED_NATURAL_RESULT.json'
    manifest = P6/'INPUTS/DATA_PRESELECTION/SELECTION/LOCKED_NATURAL.csv'
    out_dir = P7/'RESULTS/METRICS/HISTORICAL_EXPLORATION6_NATURAL'
    if not result.exists() or not manifest.exists():
        raise FileNotFoundError('Exploration-6 historical natural artifacts not found; update P6 in section 0.')
    subprocess.run([
        sys.executable, str(P7/'CODE/ANALYSIS/scripts/analyze_frozen_logits.py'),
        '--result', str(result), '--manifest', str(manifest), '--out-dir', str(out_dir)
    ], check=True)
    print('Historical replay complete. Do NOT use it to tune Exploration-7 priors/thresholds.')


4. Set the data and checkpoints

Seed 17 is the main run.
Seeds 29 and 43 are used to check if the same setup gives similar results.

Exp1 and Exp2 run seed 17 first.
Only run seeds 29 and 43 if seed 17 passes the gate.

Exp3 and Exp5 use all three seeds.
Each seed is checked separately.


In [ ]:
paths_file = P7/'INPUTS/PATHS.json'
if not paths_file.exists():
    tpl = json.loads((P7/'INPUTS/TEMPLATES/PATHS_TEMPLATE.json').read_text())
    replacements = {'<PROMPT6_ROOT>': str(P6), '<P7_ROOT>': str(P7)}
    for k, v in list(tpl.items()):
        if isinstance(v, str):
            for old, new in replacements.items():
                v = v.replace(old, new)
            tpl[k] = v
    paths_file.write_text(json.dumps(tpl, indent=2))

paths = json.loads(paths_file.read_text())
CHECKPOINT_MAP = P7/'INPUTS/EXPLORATION6_CHECKPOINTS.json'
checkpoint_payload = {
    'primary_seed': 17,
    'replication_seeds': [29, 43],
    'checkpoints': {
        '17': paths['prompt6_checkpoint_seed17'],
        '29': paths['prompt6_checkpoint_seed29'],
        '43': paths['prompt6_checkpoint_seed43'],
    },
}
CHECKPOINT_MAP.write_text(json.dumps(checkpoint_payload, indent=2))
print('Edit if needed:', paths_file)
print(paths_file.read_text())
print('Checkpoint map:', CHECKPOINT_MAP)
print(CHECKPOINT_MAP.read_text())


5. Choose which experiments to run


In [ ]:
paths = json.loads(paths_file.read_text())
RUN_MODE = str(paths.get('run_mode', 'confirmatory'))
if RUN_MODE not in {'confirmatory', 'exploratory_public_puma'}:
    raise ValueError(f'Unknown run_mode: {RUN_MODE}')
RUN_PIPELINE = True  # run Exp1-Exp5 with the bundled exploratory data
RUN_EXP1 = True
RUN_EXP2 = True
RUN_EXP3 = True
RUN_EXP4_REVIEW_PACKAGE = True
RUN_EXP5_CONTEXT = True
RUN_STAGE1_SENSITIVITY = False
STAGE_RUNTIME_LOCAL = True  # copy inputs to local the notebook storage for faster reading

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Execution device:', DEVICE)

# Run seed 17 first.
# Only run seeds 29 and 43 if seed 17 passes.
REPLICATION_POLICY = 'on_primary_pass'

RESULTS = P7/'RESULTS'/('CONFIRMATORY' if RUN_MODE == 'confirmatory' else 'EXPLORATORY_PUBLIC_PUMA')
CACHE = RESULTS/'CACHE'
METRICS = RESULTS/'METRICS'
REVIEW = RESULTS/'REVIEW'
for d in [CACHE, METRICS, REVIEW]:
    d.mkdir(parents=True, exist_ok=True)
run_mode_record = {
    'run_mode': RUN_MODE,
    'confirmatory_claims_allowed': RUN_MODE == 'confirmatory',
    'warning': None if RUN_MODE == 'confirmatory' else 'EXPLORATORY ONLY: public PUMA data were exposed during Exploration 6.',
}
(METRICS/'RUN_MODE.json').write_text(json.dumps(run_mode_record, indent=2))
print(json.dumps(run_mode_record, indent=2))

def run(cmd):
    """Run a child command while streaming its combined stdout/stderr into the notebook."""
    from collections import deque
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd), flush=True)
    tail = deque(maxlen=200)
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        tail.append(line)
        print(line, end='', flush=True)
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd, output=''.join(tail))
    return rc

def require_nonempty(*keys):
    missing = [k for k in keys if not str(paths.get(k, '')).strip()]
    if missing:
        raise ValueError('Fill these PATHS.json fields first: ' + ', '.join(missing))
    for k in keys:
        if not Path(paths[k]).exists():
            raise FileNotFoundError(f'{k}: {paths[k]}')


6. Check the input data


In [ ]:
if RUN_PIPELINE:
    require_nonempty(
        'prompt6_checkpoint_seed17', 'prompt6_checkpoint_seed29', 'prompt6_checkpoint_seed43',
        'uni2_weights',
        'prompt7_natural_train_manifest', 'prompt7_natural_calibration_manifest',
        'prompt7_natural_test_manifest'
    )
    # Check the checkpoint files for seeds 17, 29, and 43.
    # Use the paths saved in PATHS.json.
    from puma_exploration7.checkpoint import load_checkpoint_map
    from puma_exploration7.manifest import read_manifest, assert_external_group_disjoint
    validated_checkpoints = load_checkpoint_map(CHECKPOINT_MAP, require_all=True)
    print('Validated Exploration-6 replication checkpoints:', validated_checkpoints)
    _tr = read_manifest(paths['prompt7_natural_train_manifest'])
    _ca = read_manifest(paths['prompt7_natural_calibration_manifest'])
    _te = read_manifest(paths['prompt7_natural_test_manifest'])
    assert_external_group_disjoint(_tr, _ca)
    assert_external_group_disjoint(_tr, _te)
    assert_external_group_disjoint(_ca, _te)
    print('Confirmed pairwise group disjointness across train/calibration/test.')
    role_to_key = {
        'train': 'prompt7_natural_train_manifest',
        'calibration': 'prompt7_natural_calibration_manifest',
        'test': 'prompt7_natural_test_manifest',
    }
    for role, key in role_to_key.items():
        out = METRICS/f'INPUT_AUDIT_{role.upper()}.json'
        run([sys.executable, P7/'CODE/COMMON/scripts/validate_inputs.py',
             '--manifest', paths[key], '--out', out])
    print('Preflight audits written to', METRICS)
else:
    print('RUN_PIPELINE=False: experiment pipeline skipped.')


7. Prepare the files and build the cache

Copy the files needed for the experiments to a temporary working folder.
Then build the cache so the same features do not need to be created again for every experiment.

The original input files are not changed.


In [ ]:
if RUN_PIPELINE:
    ACTIVE_MANIFESTS = {
        'train': Path(paths['prompt7_natural_train_manifest']),
        'calibration': Path(paths['prompt7_natural_calibration_manifest']),
        'test': Path(paths['prompt7_natural_test_manifest']),
    }
    ACTIVE_WEIGHTS = Path(paths['uni2_weights'])
    ACTIVE_CHECKPOINT_MAP = CHECKPOINT_MAP
    ACTIVE_WEIGHTS_SHA256 = ''
    if STAGE_RUNTIME_LOCAL:
        runtime_json = Path('/content/PUMA_P7_RUNTIME/runtime_paths.json')
        run([
            sys.executable, P7/'CODE/COMMON/scripts/stage_runtime_inputs.py',
            '--train-manifest', ACTIVE_MANIFESTS['train'],
            '--calibration-manifest', ACTIVE_MANIFESTS['calibration'],
            '--test-manifest', ACTIVE_MANIFESTS['test'],
            '--weights', ACTIVE_WEIGHTS,
            '--checkpoint-map', CHECKPOINT_MAP,
            '--runtime-root', '/content/PUMA_P7_RUNTIME',
            '--out', runtime_json,
            '--images', 'auto', '--image-workers', '4',
        ])
        runtime = json.loads(runtime_json.read_text())
        ACTIVE_MANIFESTS = {k: Path(v) for k, v in runtime['manifests'].items()}
        ACTIVE_WEIGHTS = Path(runtime['uni2_weights'])
        ACTIVE_WEIGHTS_SHA256 = runtime['uni2_weights_sha256']
        ACTIVE_CHECKPOINT_MAP = Path(runtime['checkpoint_map'])
        print('Local image staging:', runtime['images_staged'])
    cmd = [
        sys.executable, P7/'CODE/TRAINING/scripts/prepare_exploration7_caches.py',
        '--train-manifest', ACTIVE_MANIFESTS['train'],
        '--calibration-manifest', ACTIVE_MANIFESTS['calibration'],
        '--test-manifest', ACTIVE_MANIFESTS['test'],
        '--weights', ACTIVE_WEIGHTS,
        '--out-dir', CACHE,
        '--batch-size', '0', '--num-workers', '-1', '--device', DEVICE,
    ]
    if ACTIVE_WEIGHTS_SHA256:
        cmd += ['--weights-sha256', ACTIVE_WEIGHTS_SHA256]
    if RUN_EXP5_CONTEXT:
        cmd += ['--with-context']
    run(cmd)
    print('Reusable Exploration-7 caches ready; UNI2 was loaded at most once in this preparation step.')
else:
    ACTIVE_MANIFESTS = {}
    ACTIVE_WEIGHTS = Path(paths.get('uni2_weights',''))
    ACTIVE_CHECKPOINT_MAP = CHECKPOINT_MAP


8. Check the three Exploration 6 checkpoints

Use the matching checkpoint for seeds 17, 29, and 43.

Seed 17 is the main run.
Seeds 29 and 43 are only used to repeat the same setup.


In [ ]:
if RUN_PIPELINE:
    checkpoint_map = json.loads(Path(ACTIVE_CHECKPOINT_MAP).read_text())
    print('Primary seed:', checkpoint_map['primary_seed'])
    print('Replication seeds:', checkpoint_map['replication_seeds'])
    for seed, path in checkpoint_map['checkpoints'].items():
        if not Path(path).is_file():
            raise FileNotFoundError(f'Exploration-6 seed {seed} checkpoint missing: {path}')
        print(f'seed {seed}: {path}')


9. H7.1 - Adjust the class prior and logits

Run seed 17 first.
If it passes the gate, repeat the same method with seeds 29 and 43.

Use only the new calibration data to estimate the target prior.
Do not tune each seed separately.


In [ ]:
if RUN_PIPELINE and RUN_EXP1:
    freeze = METRICS/'PROTOCOL_FREEZE_H7_1.json'
    if not freeze.exists():
        run([sys.executable, P7/'CODE/COMMON/scripts/freeze_protocol.py',
             '--config', P7/'CODE/CONFIGS/10_EXP1_PRIOR_CORRECTION.json', '--out', freeze])
    run([
        sys.executable, P7/'CODE/EVALUATION/scripts/run_exp1_prior_correction.py',
        '--calibration-manifest', ACTIVE_MANIFESTS['calibration'],
        '--calibration-features', CACHE/'calibration_gaussian_f96.npy',
        '--calibration-tier-a', CACHE/'calibration_tierA.npy',
        '--test-manifest', ACTIVE_MANIFESTS['test'],
        '--test-features', CACHE/'test_gaussian_f96.npy',
        '--test-tier-a', CACHE/'test_tierA.npy',
        '--checkpoint-map', ACTIVE_CHECKPOINT_MAP,
        '--out-dir', RESULTS/'EXPERIMENTS/EXP1_PRIOR_CORRECTION',
        '--source-prior', 'uniform',
        '--replication-policy', REPLICATION_POLICY,
        '--save-primary-predictions',
        '--device', DEVICE,
    ])


10. H7.2 - Retrain only the classifier on natural data

Train seed 17 first.
If it passes the gate, repeat the same setup with seeds 29 and 43.

Keep the encoder frozen.
Only retrain the classifier.


In [ ]:
if RUN_PIPELINE and RUN_EXP2:
    freeze = METRICS/'PROTOCOL_FREEZE_H7_2.json'
    if not freeze.exists():
        run([sys.executable, P7/'CODE/COMMON/scripts/freeze_protocol.py',
             '--config', P7/'CODE/CONFIGS/20_EXP2_NATURAL_CLASSIFIER.json', '--out', freeze])
    run([
        sys.executable, P7/'CODE/TRAINING/scripts/run_exp2_natural_classifier.py',
        '--train-manifest', ACTIVE_MANIFESTS['train'],
        '--train-features', CACHE/'train_gaussian_f96.npy',
        '--train-tier-a', CACHE/'train_tierA.npy',
        '--test-manifest', ACTIVE_MANIFESTS['test'],
        '--test-features', CACHE/'test_gaussian_f96.npy',
        '--test-tier-a', CACHE/'test_tierA.npy',
        '--checkpoint-map', ACTIVE_CHECKPOINT_MAP,
        '--out-dir', RESULTS/'EXPERIMENTS/EXP2_NATURAL_CRT',
        '--replication-policy', REPLICATION_POLICY,
        '--device', DEVICE,
    ])


11. H7.3 - Compare group diversity and sample density

This experiment checks whether having more independent groups is better than having more samples from the same groups.

Repeat the same experiment with seeds 17, 29, and 43.


In [ ]:
if RUN_PIPELINE and RUN_EXP3:
    freeze = METRICS/'PROTOCOL_FREEZE_H7_3.json'
    if not freeze.exists():
        run([sys.executable, P7/'CODE/COMMON/scripts/freeze_protocol.py',
             '--config', P7/'CODE/CONFIGS/30_EXP3_GROUP_DIVERSITY.json', '--out', freeze])
    run([
        sys.executable, P7/'CODE/TRAINING/scripts/run_exp3_group_diversity.py',
        '--train-manifest', ACTIVE_MANIFESTS['train'],
        '--features', CACHE/'train_gaussian_f96.npy',
        '--tier-a', CACHE/'train_tierA.npy',
        '--test-manifest', ACTIVE_MANIFESTS['test'],
        '--test-features', CACHE/'test_gaussian_f96.npy',
        '--test-tier-a', CACHE/'test_tierA.npy',
        '--checkpoint-map', ACTIVE_CHECKPOINT_MAP,
        '--out-dir', RESULTS/'EXPERIMENTS/EXP3_GROUP_DIVERSITY',
        '--group-col', 'auto', '--seeds', '17,29,43', '--device', DEVICE,
    ])


12. H7.4 - Create the annotation review set

Use the calibration data for the review set.

The reviewers only see the image crops and the blinded CSV.
Keep the model predictions and original labels in a separate key file.


In [ ]:
if RUN_PIPELINE and RUN_EXP4_REVIEW_PACKAGE:
    primary_predictions = RESULTS/'EXPERIMENTS/EXP1_PRIOR_CORRECTION/calibration_seed17.csv'
    if not primary_predictions.exists():
        raise FileNotFoundError('Exp4 needs the seed-17 frozen calibration predictions produced by Exp1.')
    run([sys.executable, P7/'CODE/EVALUATION/scripts/run_exp4_annotation_review.py',
         '--predictions', primary_predictions,
         '--manifest', ACTIVE_MANIFESTS['calibration'],
         '--out-dir', REVIEW/'EXP4_BLINDED', '--per-pair', '30'])
    print('Give REVIEW/EXP4_BLINDED/review_blinded.csv + crops/ to reviewers; keep review_key.csv hidden.')


13. H7.5 - Compare small and large image context

Compare FOV96 and FOV192 to see whether a larger image area gives useful extra information.

Repeat the same test with split seeds 17, 29, and 43.


In [ ]:
if RUN_PIPELINE and RUN_EXP5_CONTEXT:
    freeze = METRICS/'PROTOCOL_FREEZE_H7_5.json'
    if not freeze.exists():
        run([sys.executable, P7/'CODE/COMMON/scripts/freeze_protocol.py',
             '--config', P7/'CODE/CONFIGS/50_EXP5_CONTEXT_PROBE.json', '--out', freeze])
    run([sys.executable, P7/'CODE/TRAINING/scripts/run_exp5_context_probe.py',
         '--manifest', ACTIVE_MANIFESTS['train'],
         '--local-features', CACHE/'train_gaussian_f96.npy',
         '--context-features', CACHE/'train_gaussian_f192.npy',
         '--out', METRICS/'EXP5_CONTEXT_PROBE.json',
         '--group-col', 'auto', '--seeds', '17,29,43'])


14. H7.6 - Check Stage-1 localization sensitivity

Only run this part when the matching Stage-1 proposal files are available.


In [ ]:
if RUN_STAGE1_SENSITIVITY:
    require_nonempty('prompt7_stage1_manifest','prompt7_stage1_predictions')
    run([sys.executable, P7/'CODE/EVALUATION/scripts/run_exp6_stage1_sensitivity.py',
         '--manifest', paths['prompt7_stage1_manifest'],
         '--predictions', paths['prompt7_stage1_predictions'],
         '--out', METRICS/'EXP6_STAGE1_SENSITIVITY.csv'])
else:
    print('Stage-1 sensitivity skipped until certified matched proposal artifacts are available.')


15. Summarize the Exploration 7 results


In [ ]:
run([sys.executable, P7/'CODE/ANALYSIS/scripts/summarize_exploration7.py', '--results', RESULTS])
print('Read README.md and REPORTS/METHODS/DETAILED_PSEUDOCODE_AND_DEBATE.md before interpreting promotion gates.')


In [ ]:
from google.colab import runtime
runtime.unassign()